In [ ]:
import kagglehub
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

In [ ]:
# 1. Descargar Dataset
path = kagglehub.dataset_download("jhunbrianandam/quickdraw10-dataset")
data_root = os.path.join(path, "data")

# 2. Definir Transformaciones (ResNet Standard)
# Nota: Usamos la misma transform para train y test para simplificar
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3), # Forzar 3 canales
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. Cargar todo el dataset con ImageFolder
full_dataset = datasets.ImageFolder(root=data_root, transform=transform)
classes = full_dataset.classes
print(f"Clases: {classes}")

# --- AQUÍ USAMOS SCIKIT-LEARN ---

# Obtenemos las etiquetas de todo el dataset para poder estratificar
targets = full_dataset.targets 

# Generamos los ÍNDICES de entrenamiento y prueba
train_idx, test_idx = train_test_split(
    np.arange(len(full_dataset)), # Array de 0 a N-1
    test_size=0.2,                # 20% para test
    shuffle=True,
    stratify=targets,             # ¡Importante! Mantiene el balance de clases
    random_state=42               # Para reproducibilidad
)

# Creamos los subsets de PyTorch usando esos índices
train_dataset = Subset(full_dataset, train_idx)
test_dataset = Subset(full_dataset, test_idx)

# Creamos los DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Total imágenes: {len(full_dataset)}")
print(f"Entrenamiento: {len(train_dataset)} imágenes")
print(f"Prueba (Test): {len(test_dataset)} imágenes")

Buscando imágenes en: /home/klefur/.cache/kagglehub/datasets/jhunbrianandam/quickdraw10-dataset/versions/1/data
Clases detectadas: ['airplane', 'bat', 'broccoli', 'banana', 'angel', 'apple', 'bird', 'bowtie', 'book', 'ant']


In [ ]:
# 1. Descargar modelo pre-entrenado
model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)

# 2. CONGELAR todos los parámetros (Backbone)
for param in model.parameters():
    param.requires_grad = False

# 3. Modificar la capa final (Head)
# Esta es la única capa que tendrá requires_grad = True por defecto al ser creada nueva
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(classes))

model = model.to(device)

Entrenando en: cpu
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /home/klefur/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:04<00:00, 21.6MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

epochs = 3

print("Iniciando entrenamiento...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Acc Train: {epoch_acc:.2f}%")

print("Entrenamiento finalizado.")

Epoch 1/3 en proceso...


In [ ]:
model.eval() # Modo evaluación (apaga Dropout, Batchnorm fijo, etc)
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad(): # No calculamos gradientes para test
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # Guardamos para reporte
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(f"Precisión en Test Set: {100 * correct / total:.2f}%")

In [ ]:
# Generar reporte de texto
print("Reporte de Clasificación:")
print(classification_report(all_labels, all_preds, target_names=classes))

# Matriz de Confusión Visual
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de Confusión QuickDraw')
plt.show()